# HTTP basics

**Objective:** Build validated request and response boundaries with explicit methods and status codes.

## Simple version

In [ ]:
import httpx
from fastapi import FastAPI, status


app = FastAPI()


@app.get("/items/{item_id}")
async def get_item(item_id: int) -> dict:
    return {"id": item_id}


@app.post("/items", status_code=status.HTTP_201_CREATED)
async def create_item() -> dict:
    return {"id": 1}


transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    get_response = await client.get("/items/7")
    post_response = await client.post("/items")

print(get_response.status_code, post_response.status_code)

## Polished version

In [ ]:
import httpx
from fastapi import APIRouter, FastAPI, status
from pydantic import BaseModel, Field


class ItemCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100)


class ItemResponse(BaseModel):
    id: int
    name: str


def create_app() -> FastAPI:
    router = APIRouter(prefix="/items")

    @router.get("/{item_id}", response_model=ItemResponse)
    async def get_item(item_id: int) -> ItemResponse:
        return ItemResponse(id=item_id, name="book")

    @router.post("", response_model=ItemResponse, status_code=status.HTTP_201_CREATED)
    async def create_item(body: ItemCreate) -> ItemResponse:
        return ItemResponse(id=1, name=body.name)

    app = FastAPI()
    app.include_router(router)
    return app


app = create_app()
transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    response = await client.post("/items", json={"name": "notebook"})

print(response.status_code, response.json())